# Stability of Linear Systems and the Condition Number
### Numerical Linear Algebra — Practice 2 (guided, self-contained solution)

This notebook walks through the whole practice: the theory behind the condition number of a matrix, and the four problems of the handout, each one **explained, implemented, executed and interpreted**.

It is written for a reader with a solid mathematical background (MSc level): statements are precise, the key results are *derived*, and every numerical experiment is tied back to a theorem.

## Contents

| Part | Topic |
|---|---|
| **0** | Theory: norms, condition number, perturbation bound, backward vs forward error |
| **1** | Problem 1 — Ill-conditioned matrices of small size: (a) Vandermonde, (b) Hilbert & Szegő's asymptotics |
| **2** | Problem 2 — Condition number vs accuracy of the computed solution (Frank matrix) |
| **3** | Problem 3 — "Never compute $A^{-1}$": inverse method vs GEPP (stability and cost) |
| **4** | Problem 4 — Estimating $\kappa_1(A)$ without inverting: Hager's algorithm |
| **5** | Summary and further reading |

**Dependencies:** `numpy`, `scipy`, `matplotlib`, and (optionally, for "exact" reference values) `mpmath`.
If something is missing: `pip install numpy scipy matplotlib mpmath`.

In [ ]:
import time
import numpy as np
import scipy.linalg as sla
import matplotlib.pyplot as plt

try:
    import mpmath as mp
    HAVE_MP = True
except ImportError:                      # the notebook still runs, only the "exact" reference curves are skipped
    HAVE_MP = False
    print("mpmath not found: high-precision reference values will be skipped (pip install mpmath).")

np.set_printoptions(precision=4, suppress=False, linewidth=130)
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

EPS = np.finfo(float).eps          # machine epsilon  2^-52 ~ 2.2e-16
U   = EPS / 2                      # unit roundoff    2^-53 ~ 1.1e-16 (round-to-nearest)
print(f"machine epsilon = {EPS:.3e},  unit roundoff u = {U:.3e}")

# Part 0 — Theory

## 0.1 Why roundoff matters: the three sources of error

When we compute $\hat x$ on a computer for the system $Ax=b$, three things happen:

1. **Data errors.** The entries of $A$ and $b$ are stored in floating point (or come from measurements), so we are really working with $A+\delta A$ and $b+\delta b$, with $\|\delta A\|/\|A\| = O(\varepsilon_{\rm mach})$.
2. **Algorithmic rounding errors.** Every flop of the elimination is rounded.
3. **Sensitivity of the problem itself.** Even an *exact* solution of the perturbed system can be far from $x$ if $A$ is ill-conditioned.

The first and third items are properties of the **problem** (quantified by the condition number); the second is a property of the **algorithm** (quantified by backward stability). The central idea of backward error analysis is to keep them separate:

$$
\underbrace{\text{forward error}}_{\|\hat x - x\|/\|x\|}\;\lesssim\;\underbrace{\kappa(A)}_{\text{problem}}\times\underbrace{\text{backward error}}_{\text{algorithm}} .
$$

## 0.2 Matrix norms and the condition number

For a vector norm $\|\cdot\|$, the induced matrix norm is $\|A\|=\max_{x\neq0}\|Ax\|/\|x\|$. The ones used in this practice:

* $\|A\|_1=\max_j\sum_i|a_{ij}|$ (max column sum),
* $\|A\|_\infty=\max_i\sum_j|a_{ij}|$ (max row sum),
* $\|A\|_2=\sigma_{\max}(A)$ (largest singular value).

For a nonsingular $A$, the **condition number** is
$$
\kappa(A)=\|A\|\,\|A^{-1}\| .
$$

**Basic properties** (all for induced norms):

* $\kappa(A)\ge \|AA^{-1}\|=1$.
* $\kappa(cA)=\kappa(A)$ for $c\neq0$: it is scale invariant (so it is *not* related to the determinant; $\det(10^{-10}I_n)$ is tiny but $\kappa=1$).
* $\kappa(A)=\kappa(A^{-1})$.
* $\kappa_2(A)=\sigma_{\max}/\sigma_{\min}$, and $\kappa_2(QA)=\kappa_2(AQ)=\kappa_2(A)$ for orthogonal $Q$.
* Norm equivalence: $\tfrac1n\kappa_2(A)\le\kappa_1(A)\le n\,\kappa_2(A)$, so all the usual condition numbers agree up to a factor $\le n$ — to estimate the *order of magnitude* any of them will do.

In [ ]:
# Quick numerical sanity check of the properties above
rng = np.random.default_rng(0)
n = 8
A = rng.standard_normal((n, n))

def kappa(A, p):                       # kappa_p(A) = ||A||_p ||A^{-1}||_p  (direct, via the inverse)
    return np.linalg.norm(A, p) * np.linalg.norm(np.linalg.inv(A), p)

s = np.linalg.svd(A, compute_uv=False)
print("kappa_2 via norms        :", kappa(A, 2))
print("kappa_2 = sigma_1/sigma_n:", s[0] / s[-1])
print("scale invariance         :", kappa(1e-10 * A, 2))
print("kappa(A) = kappa(A^-1)   :", kappa(np.linalg.inv(A), 2))
k1, k2 = kappa(A, 1), kappa(A, 2)
print(f"norm equivalence  k2/n = {k2/n:.3f} <= k1 = {k1:.3f} <= n*k2 = {n*k2:.3f}")

## 0.3 The perturbation bound (derivation of the formula in the handout)

Let $Ax=b$ and suppose $\hat x=x+\delta x$ solves the perturbed system
$$
(A+\delta A)\hat x=b+\delta b .
$$
Subtracting $Ax=b$ we get $A\,\delta x=\delta b-\delta A\,\hat x$, i.e.
$$
\delta x=A^{-1}\bigl(\delta b-\delta A\,\hat x\bigr)
\;\Longrightarrow\;
\|\delta x\|\le\|A^{-1}\|\bigl(\|\delta b\|+\|\delta A\|\,\|\hat x\|\bigr).
$$
Dividing by $\|\hat x\|$ and inserting $\|A\|/\|A\|=1$:
$$
\boxed{\;\frac{\|\delta x\|}{\|\hat x\|}\;\le\;\kappa(A)\left(\frac{\|\delta A\|}{\|A\|}+\frac{\|\delta b\|}{\|A\|\,\|\hat x\|}\right)\;}
$$
which is exactly the formula quoted in the handout. Since $\|b\|\le\|A\|\|x\|$, the second term is at least as large as $\|\delta b\|/\|b\|$ up to $O(\delta)$ corrections, and to first order one often writes
$$
\frac{\|\delta x\|}{\|x\|}\lesssim\kappa(A)\left(\frac{\|\delta A\|}{\|A\|}+\frac{\|\delta b\|}{\|b\|}\right).
$$

**Reading the bound.** If the data are perturbed by a relative amount $\varepsilon\approx10^{-16}$ then we should expect to lose about $\log_{10}\kappa(A)$ digits:
$$
\text{correct digits in }\hat x\;\approx\;16-\log_{10}\kappa(A).
$$

### The bound is sharp
It is not just a pessimistic estimate: for every $A$ there are $b,\delta b$ attaining it. Take the SVD $A=\sum_i\sigma_iu_iv_i^T$ and choose
$$
b=\sigma_1u_1\;(\Rightarrow x=v_1),\qquad \delta b=\varepsilon\,u_n\;(\Rightarrow\delta x=\tfrac{\varepsilon}{\sigma_n}v_n).
$$
Then $\|\delta b\|/\|b\|=\varepsilon/\sigma_1$ and $\|\delta x\|/\|x\|=\varepsilon/\sigma_n$, so the ratio of relative errors is exactly $\sigma_1/\sigma_n=\kappa_2(A)$.

In [ ]:
# Worst-case perturbation: the bound is attained
rng = np.random.default_rng(1)
n = 6
U_, s, Vt = np.linalg.svd(rng.standard_normal((n, n)) @ np.diag([1, 1, 1, 1e-2, 1e-4, 1e-6]))
A = U_ @ np.diag(s) @ Vt

b  = s[0] * U_[:, 0]                 # x = v_1
db = 1e-10 * U_[:, -1]               # worst direction
x  = np.linalg.solve(A, b)
dx = np.linalg.solve(A, b + db) - x

print("kappa_2(A)                    = %.4e" % (s[0] / s[-1]))
print("relative error in b           = %.4e" % (np.linalg.norm(db) / np.linalg.norm(b)))
print("relative error in x           = %.4e" % (np.linalg.norm(dx) / np.linalg.norm(x)))
print("ratio (amplification factor)  = %.4e   <-- equals kappa_2" % ((np.linalg.norm(dx) / np.linalg.norm(x)) / (np.linalg.norm(db) / np.linalg.norm(b))))

## 0.4 Geometric meaning: distance to singularity

Let $A=U\Sigma V^T$. By the Eckart–Young theorem, the closest singular matrix to $A$ in the 2-norm is obtained by deleting the smallest singular value:
$$
\min\{\|\delta A\|_2:\;A+\delta A\text{ singular}\}=\sigma_n
\quad\Longleftrightarrow\quad
\min\Bigl\{\tfrac{\|\delta A\|_2}{\|A\|_2}:\;A+\delta A\text{ singular}\Bigr\}=\frac1{\kappa_2(A)} .
$$
So **$\kappa_2(A)$ is the reciprocal of the relative distance to the set of singular matrices**: a matrix with $\kappa_2=10^{16}$ is, relative to its size, within roundoff of a singular matrix. That is exactly what "ill-conditioned" means; it also explains why the determinant is irrelevant.

In [ ]:
# A relative perturbation of size 1/kappa_2 makes A exactly singular
rng = np.random.default_rng(2)
n = 6
U_, s, Vt = np.linalg.svd(rng.standard_normal((n, n)))
s = np.logspace(0, -7, n)                       # prescribe the singular values
A = (U_ * s) @ Vt

E = -s[-1] * np.outer(U_[:, -1], Vt[-1, :])     # rank-one perturbation killing sigma_n
print("relative size of the perturbation ||E||/||A|| = %.3e" % (np.linalg.norm(E, 2) / np.linalg.norm(A, 2)))
print("1/kappa_2(A)                                   = %.3e" % (s[-1] / s[0]))
print("smallest singular value of A+E                 = %.3e   (numerically zero)" % np.linalg.svd(A + E, compute_uv=False)[-1])

## 0.5 A first example: a system where tiny changes matter

$$
A=\begin{pmatrix}1&1\\1&1.0001\end{pmatrix},\qquad b=\begin{pmatrix}2\\2.0001\end{pmatrix}\;\Rightarrow\;x=\begin{pmatrix}1\\1\end{pmatrix}.
$$
The two rows describe two almost parallel lines, so their intersection moves a lot if one line is nudged. Changing $b_2$ from $2.0001$ to $2.0002$ (a relative change of $5\cdot10^{-5}$) gives the solution $(0,2)^T$: a relative change of $100\%$ in $x$.

In [ ]:
A = np.array([[1.0, 1.0], [1.0, 1.0001]])
b = np.array([2.0, 2.0001])
bp = np.array([2.0, 2.0002])

x, xp = np.linalg.solve(A, b), np.linalg.solve(A, bp)
print("kappa_2(A)         =", np.linalg.cond(A))
print("x  =", x, "   x' =", xp)
print("relative change in b:", np.linalg.norm(bp - b) / np.linalg.norm(b))
print("relative change in x:", np.linalg.norm(xp - x) / np.linalg.norm(x))
print("bound kappa*rel(b)  :", np.linalg.cond(A) * np.linalg.norm(bp - b) / np.linalg.norm(b))

## 0.6 Forward error, backward error, and the role of the algorithm

For a computed solution $\hat x$ define:

* **Forward error:** $\|\hat x-x\|/\|x\|$ — what we ultimately care about, but unknown in practice.
* **Residual:** $r=A\hat x-b$ — computable.
* **Normwise relative backward error** (Rigal–Gaches theorem):
$$
\eta_{A,b}(\hat x)=\min\Bigl\{\varepsilon:\;(A+\delta A)\hat x=b+\delta b,\;\|\delta A\|\le\varepsilon\|A\|,\;\|\delta b\|\le\varepsilon\|b\|\Bigr\}
=\frac{\|r\|}{\|A\|\,\|\hat x\|+\|b\|}.
$$
That closed-form expression is the quantity $\eta_{A,b}$ used in Problem 3.

An algorithm is **backward stable** when $\eta_{A,b}(\hat x)=O(u)$ for every input: the computed $\hat x$ is the *exact* solution of a nearby problem. Combining with the perturbation bound:
$$
\boxed{\;\text{forward error}\;\lesssim\;\kappa(A)\cdot\eta_{A,b}(\hat x)\;}
$$
* Gaussian elimination with partial pivoting (GEPP) is backward stable in practice ($\eta\approx u$; in theory $\eta\lesssim c\,n\,\rho_n u$ with a growth factor $\rho_n\le2^{n-1}$ that is tiny for virtually all matrices that occur).
* Hence **with GEPP the accuracy of $\hat x$ is governed by $\kappa(A)$ alone**. Problems 1 and 2 study $\kappa$ and its consequences; Problem 3 compares algorithms with different backward errors; Problem 4 is about computing $\kappa$ cheaply.

### Experiment: "digits lost $=\log_{10}\kappa$"
We solve $Ax=b$ for random matrices with prescribed condition number $\kappa$ (constructed through the SVD, $A=U\,\mathrm{diag}(\sigma)\,V^T$ with random orthogonal $U,V$ and singular values between $1$ and $1/\kappa$), and compare the forward error with $\kappa u$.

In [ ]:
def random_orthogonal(n, rng):
    '''Haar-distributed orthogonal matrix (QR of a Gaussian matrix with sign correction).'''
    Q, R = np.linalg.qr(rng.standard_normal((n, n)))
    return Q * np.sign(np.diag(R))

def rand_matrix_cond(n, kappa, rng):
    '''Random general matrix with kappa_2(A)=kappa and singular values log-uniformly spread in [1/kappa, 1].'''
    e = rng.uniform(0, 1, n); e[0], e[-1] = 0.0, 1.0            # exponents; guarantees sigma_max=1, sigma_min=1/kappa
    s = kappa ** (-e)
    return (random_orthogonal(n, rng) * s) @ random_orthogonal(n, rng).T

rng = np.random.default_rng(3)
n = 40
kappas = 10.0 ** np.arange(0, 17)
fwd = []
for k in kappas:
    errs = []
    for _ in range(20):
        A = rand_matrix_cond(n, k, rng)
        x0 = rng.standard_normal(n)
        b = A @ x0
        x = np.linalg.solve(A, b)
        errs.append(np.linalg.norm(x - x0) / np.linalg.norm(x0))
    fwd.append(np.mean(errs))

plt.loglog(kappas, fwd, "o-", label="mean forward error (20 systems)")
plt.loglog(kappas, U * kappas, "k--", label=r"$u\,\kappa_2(A)$")
plt.loglog(kappas, np.minimum(U * kappas, 1), alpha=0)       # keep axes tidy
plt.xlabel(r"$\kappa_2(A)$"); plt.ylabel("relative error"); plt.legend(); plt.title("Forward error of GEPP vs. condition number")
plt.show()

The forward error tracks the line $u\,\kappa_2(A)$ (typically a bit below it: the bound is a worst case) until $\kappa_2\approx1/u$, where all significant digits are lost and the relative error saturates at $O(1)$.

# Part 1 — Problem 1: Ill-conditioned matrices of small dimension

Ill-conditioning is not exclusive to huge matrices. Two classical families become numerically singular already for $n\approx10$–$20$.

> **A technical remark we must keep in mind.** If $\kappa_2(A)\gtrsim1/\varepsilon\approx10^{16}$, then a floating-point SVD cannot resolve $\sigma_{\min}$: its backward error is $O(\varepsilon\,\sigma_{\max})$, so the computed $\sigma_{\min}$ is garbage once it is below $\varepsilon\,\sigma_{\max}$, and `np.linalg.cond` **saturates** (or fluctuates) at a value of order $10^{16}$–$10^{19}$. To study the *true* growth we therefore compute $\kappa_2$ with **arbitrary-precision arithmetic** (`mpmath`), using enough digits ($\gtrsim\log_{10}\kappa+$ margin). Both curves are shown so that the saturation phenomenon is visible.

## 1(a) The Vandermonde matrix on equispaced nodes

$$
V_n=\begin{pmatrix}1&1&\cdots&1\\\alpha_1&\alpha_2&\cdots&\alpha_n\\\vdots&\vdots&&\vdots\\\alpha_1^{n-1}&\alpha_2^{n-1}&\cdots&\alpha_n^{n-1}\end{pmatrix},\qquad
\alpha_j=1-\frac{2(j-1)}{n-1},\;j=1,\dots,n,
$$
i.e. $n$ equispaced nodes in $[-1,1]$. $V_n^T$ is the matrix of the monomial basis $\{1,x,\dots,x^{n-1}\}$ evaluated at the nodes, i.e. the matrix of the *polynomial interpolation* problem: solving $V_n^Tc=y$ gives the coefficients of the interpolant. It has the same singular values as $V_n$, so $\kappa_2(V_n)=\kappa_2(V_n^T)$.

**Why do we expect ill-conditioning?** On $[-1,1]$ the monomials $x^k$ and $x^{k+2}$ look almost the same (they are both $\approx$ "flat in the middle, steep at the ends"), so the columns of the evaluation matrix are nearly linearly dependent.

In [ ]:
def alphas(n):
    j = np.arange(1, n + 1)
    return 1 - 2 * (j - 1) / (n - 1)

def vandermonde(n):
    '''V_n(i,j) = alpha_j^(i-1)  (rows = powers, as in the handout).'''
    return np.vander(alphas(n), increasing=True).T

print(vandermonde(4))

In [ ]:
def kappa2_mp(M):
    '''kappa_2 of an mpmath matrix through its (arbitrary precision) SVD.'''
    S = mp.svd_r(M, compute_uv=False)
    s_max, s_min = max(S), min(S)
    return s_max / s_min

def vandermonde_mp(n, nodes=None):
    if nodes is None:
        nodes = [1 - mp.mpf(2) * j / (n - 1) for j in range(n)]
    return mp.matrix([[nodes[j] ** i for j in range(n)] for i in range(n)])

ns = np.arange(2, 51)
k_float = np.array([np.linalg.cond(vandermonde(n)) for n in ns])

if HAVE_MP:
    k_exact = []
    for n in ns:
        mp.mp.dps = 60 + 2 * int(n)            # enough digits for kappa up to ~1e40 and beyond
        k_exact.append(float(kappa2_mp(vandermonde_mp(int(n)))))
    k_exact = np.array(k_exact)
else:
    k_exact = k_float

print(f"{'n':>3} {'kappa_2 (float64 SVD)':>24} {'kappa_2 (high precision)':>26}")
for n in [2, 5, 10, 15, 20, 25, 30, 40, 50]:
    i = n - 2
    print(f"{n:>3} {k_float[i]:>24.4e} {k_exact[i]:>26.4e}")

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(ns, k_exact, "o-", ms=4, label=r"$\kappa_2(V_n)$ (high precision)")
ax.semilogy(ns, k_float, "s--", ms=3, alpha=.7, label=r"$\kappa_2(V_n)$ (float64 SVD)")
ax.axhline(1 / EPS, color="gray", ls=":", label=r"$1/\varepsilon_{mach}$")
ax.set_xlabel("n"); ax.set_ylabel(r"$\kappa_2$"); ax.set_title("Vandermonde matrix, equispaced nodes in [-1,1]"); ax.legend()
plt.show()

# Fit log10(kappa) = a*n + b  (exponential growth <=> straight line in a semilog plot)
mask = ns >= 10
a, b = np.polyfit(ns[mask], np.log10(k_exact[mask]), 1)
print(f"log10 kappa_2(V_n) ~ {a:.4f} n + {b:.3f}   =>   kappa_2(V_n) ~ {10**b:.3g} * {10**a:.3f}^n ,   ln-rate = {a*np.log(10):.3f}")
print("first n with kappa_2 > 1/eps (exact):", ns[np.argmax(k_exact > 1 / EPS)])

### Interpretation

* In a semilog plot $\kappa_2(V_n)$ is (asymptotically) a **straight line**: the condition number grows **exponentially** in $n$, by a roughly constant factor (fit printed above) for each additional node.
* It crosses $1/\varepsilon\approx4.5\cdot10^{15}$ for $n$ around 20: from there on, solving a Vandermonde system in double precision returns **no correct digit** — even though the system is only $20\times20$ and perfectly nonsingular (indeed $\det V_n=\prod_{i<j}(\alpha_j-\alpha_i)\neq0$).
* The float64 SVD curve follows the exact one until about $10^{16}$ and then **stops tracking** (saturation, as announced above).

### Extension: it is the *basis* and the *nodes* that are bad, not the problem
Interpolation itself is fine; the monomial basis on equispaced nodes is a terrible representation. Compare three choices of (basis, nodes), all for degree $n-1$ polynomials:

1. monomials, equispaced nodes (the exercise);
2. monomials, Chebyshev nodes $\cos\frac{(2j-1)\pi}{2n}$;
3. **Chebyshev polynomials $T_k$ at Chebyshev nodes** — here the evaluation matrix has orthogonal columns (discrete orthogonality), so $\kappa_2=\sqrt2$ essentially for any $n$.

In [ ]:
from numpy.polynomial import chebyshev as C

ns2 = np.arange(2, 41)
k_mono_equi = np.array([float(kappa2_mp(vandermonde_mp(int(n)))) if not HAVE_MP or True else 0 for n in ns2]) if HAVE_MP else None

k_mono_cheb, k_cheb_cheb = [], []
for n in ns2:
    n = int(n)
    if HAVE_MP:
        mp.mp.dps = 60 + 2 * n
        nodes = [mp.cos((2 * j - 1) * mp.pi / (2 * n)) for j in range(1, n + 1)]
        k_mono_cheb.append(float(kappa2_mp(vandermonde_mp(n, nodes))))
    xs = np.cos((2 * np.arange(1, n + 1) - 1) * np.pi / (2 * n))
    k_cheb_cheb.append(np.linalg.cond(C.chebvander(xs, n - 1)))

if HAVE_MP:
    plt.semilogy(ns2, k_mono_equi, "o-", ms=3, label="monomials, equispaced nodes")
    plt.semilogy(ns2, k_mono_cheb, "s-", ms=3, label="monomials, Chebyshev nodes")
plt.semilogy(ns2, k_cheb_cheb, "^-", ms=3, label=r"Chebyshev basis $T_k$, Chebyshev nodes")
plt.xlabel("n"); plt.ylabel(r"$\kappa_2$"); plt.legend(); plt.title("Conditioning of polynomial interpolation matrices")
plt.show()
print("kappa_2 of the Chebyshev/Chebyshev matrix for n=40:", k_cheb_cheb[-1], "  (sqrt(2) = %.4f)" % np.sqrt(2))

Moral: a good choice of basis and nodes turns an exponentially ill-conditioned problem into a perfectly conditioned one. *Conditioning is a property of the formulation.*

## 1(b) The Hilbert matrix and Szegő's asymptotics

$$
H_n(i,j)=\frac1{i+j-1},\qquad 1\le i,j\le n.
$$
$H_n$ is symmetric positive definite (it is the Gram matrix of the monomials in $L^2(0,1)$: $H_{ij}=\int_0^1x^{i-1}x^{j-1}dx$), hence $\kappa_2(H_n)=\lambda_{\max}/\lambda_{\min}$. The claim to check (Szegő) is
$$
\kappa_2(H_n)\sim e^{3.5\,n}.
$$
More precisely the rate is $4\ln(1+\sqrt2)\approx3.5255$, and the refined asymptotics has the form $\kappa_2(H_n)\approx C\,(1+\sqrt2)^{4n}/\sqrt n$.

**Strategy.** (i) Compute $\kappa_2(H_n)$ in high precision for $n=1,\dots,40$; (ii) fit the slope of $\ln\kappa_2(H_n)$ against $n$ (it must be $\approx3.5$); (iii) check that the *sub-exponential* prefactor stabilizes; (iv) show where float64 breaks down.

In [ ]:
ns_h = np.arange(1, 41)
kh_float = np.array([np.linalg.cond(sla.hilbert(int(n))) for n in ns_h])

if HAVE_MP:
    kh_exact = []
    for n in ns_h:
        n = int(n)
        mp.mp.dps = 40 + 3 * n
        E = mp.eigsy(mp.hilbert(n), eigvals_only=True)      # H_n is symmetric positive definite
        kh_exact.append(float(max(E) / min(E)))
    kh_exact = np.array(kh_exact)
else:
    kh_exact = kh_float

print(f"{'n':>3} {'kappa_2 (float64)':>20} {'kappa_2 (high prec.)':>22} {'e^(3.5 n)':>14}")
for n in [1, 2, 3, 5, 8, 10, 12, 15, 20, 30, 40]:
    print(f"{n:>3} {kh_float[n-1]:>20.4e} {kh_exact[n-1]:>22.4e} {np.exp(3.5*n):>14.3e}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))

ax = axs[0]
ax.semilogy(ns_h, kh_exact, "o-", ms=4, label=r"$\kappa_2(H_n)$ (high precision)")
ax.semilogy(ns_h, kh_float, "s--", ms=3, alpha=.7, label=r"$\kappa_2(H_n)$ (float64)")
ax.semilogy(ns_h, np.exp(3.5 * ns_h), "k:", label=r"$e^{3.5n}$")
ax.axhline(1 / EPS, color="gray", ls=":"); ax.text(1, 1.5 / EPS, r"$1/\varepsilon$", color="gray")
ax.set_xlabel("n"); ax.set_title("Hilbert matrix"); ax.legend()

# slope of ln(kappa) vs n
ax = axs[1]
ax.plot(ns_h, np.log(kh_exact), "o-", ms=4, label=r"$\ln\kappa_2(H_n)$")
ax.plot(ns_h, 3.5 * ns_h, "k:", label=r"$3.5\,n$")
ax.set_xlabel("n"); ax.set_title("Logarithmic scale"); ax.legend()
plt.show()

m = ns_h >= 15
slope, intercept = np.polyfit(ns_h[m], np.log(kh_exact[m]), 1)
print(f"fitted slope of ln(kappa_2) for n>=15 : {slope:.4f}")
print(f"Szego rate 4*ln(1+sqrt2)              : {4*np.log(1+np.sqrt(2)):.4f}      (the handout's e^(3.5 n) is the rounded version)")

In [ ]:
# Prefactor test:  C_n = kappa_2(H_n) * sqrt(n) / (1+sqrt2)^(4n)   must converge to a constant
rate = (1 + np.sqrt(2)) ** 4
Cn = kh_exact * np.sqrt(ns_h) / rate ** ns_h
for n in [5, 10, 20, 30, 40]:
    print(f"n={n:>2}:  C_n = {Cn[n-1]:.5f}")

### Interpretation

* The fitted slope of $\ln\kappa_2(H_n)$ is $\approx3.51$, in agreement with Szegő's $e^{3.5n}$ (the exact rate is $3.5255$; the slight difference is the sub-exponential factor $1/\sqrt n$, which contributes $-\tfrac12\ln n$ to the logarithm).
* The normalized sequence $C_n=\kappa_2(H_n)\sqrt n/(1+\sqrt2)^{4n}$ **converges** (slowly, with $O(1/n)$ corrections), confirming the form of the asymptotics, not just the exponential rate.
* The float64 computation of $\kappa_2(H_n)$ is correct only up to $n\approx12$ ($\kappa_2\approx10^{16}$) and then **saturates around $10^{18}$**–$10^{19}$, though the true value keeps growing: $\kappa_2(H_{40})\approx10^{59}$.
* Practical meaning: with $n=13$ the Hilbert matrix is numerically singular in double precision. Solving $H_nx=b$ returns garbage for $n\gtrsim13$, although $\det H_n\neq0$ for all $n$ (and $H_n^{-1}$ has *integer* entries, given by an explicit formula).

# Part 2 — Problem 2: Condition number vs. accuracy of the solution

## Statement
Consider the $n$-dimensional **Frank matrix**
$$
A_{i,j}=\begin{cases}0,&j<i-1,\\ n+1-i,&j=i-1,\\ n+1-j,&j\ge i,\end{cases}
$$
stored in double precision. Let $x_0=(1,\dots,1)^T$ and $b=Ax_0$. For $n=2,\dots,24$ compute $\kappa_2(A)$ and compare $\varepsilon_{mach}\,\kappa_2(A)$ with the relative error $e_{rel}(x)=\|x-x_0\|_2/\|x_0\|_2$ of the computed solution. Explain the results with the backward error formula.

The Frank matrix is upper Hessenberg, with $\det A=1$ for every $n$: *the determinant tells us nothing* about conditioning — yet its eigenvalues are extremely ill-conditioned and $\kappa_2$ grows super-exponentially.

A remark on the "roundoff in $b$": for this matrix all entries are small integers, so $A$ and $b=Ax_0$ are computed **exactly** in floating point. The perturbation entering the bound is therefore the one produced by the *algorithm* (backward error of GEPP: $\hat x$ solves $(A+\delta A)\hat x=b$ with $\|\delta A\|/\|A\|=O(u)$). We will also run a variant in which $b$ is explicitly polluted at the level of $\varepsilon_{mach}$, exactly as the handout suggests, and compare.

In [ ]:
def frank(n):
    i = np.arange(1, n + 1)[:, None]
    j = np.arange(1, n + 1)[None, :]
    return np.where(j >= i, n + 1 - j, np.where(j == i - 1, n + 1 - i, 0)).astype(float)

print(frank(5))
print("det(Frank_n) for n=2..8:", [round(np.linalg.det(frank(n)), 6) for n in range(2, 9)])

In [ ]:
def kappa2_frank_exact(n):
    mp.mp.dps = 100
    A = mp.matrix(frank(n).tolist())            # entries are small integers: exactly representable
    return float(kappa2_mp(A))

rng = np.random.default_rng(10)
rows = []
for n in range(2, 25):
    A = frank(n)
    x0 = np.ones(n)
    b = A @ x0                                   # exact: integer arithmetic below 2^53
    x = np.linalg.solve(A, b)                    # LAPACK gesv = LU with partial pivoting
    e_rel = np.linalg.norm(x - x0) / np.linalg.norm(x0)

    # variant: b polluted at the level of machine precision (relative perturbation <= eps in each component)
    bp = b * (1 + EPS * rng.uniform(-1, 1, n))
    xp = np.linalg.solve(A, bp)
    e_rel_p = np.linalg.norm(xp - x0) / np.linalg.norm(x0)

    k_fl = np.linalg.cond(A)
    k_ex = kappa2_frank_exact(n) if HAVE_MP else k_fl
    eta = np.linalg.norm(A @ x - b) / (np.linalg.norm(A, 2) * np.linalg.norm(x) + np.linalg.norm(b))
    rows.append((n, k_fl, k_ex, EPS * k_ex, e_rel, e_rel_p, eta))

print(f"{'n':>3} {'k2 float64':>11} {'k2 exact':>11} {'eps*k2':>10} {'e_rel':>10} {'e_rel(b pert.)':>15} {'backward err':>13} {'e_rel/(eps k2)':>15}")
for n, kf, ke, ek, e, ep, eta in rows:
    print(f"{n:>3} {kf:>11.2e} {ke:>11.2e} {ek:>10.2e} {e:>10.2e} {ep:>15.2e} {eta:>13.2e} {e/ek:>15.2e}")

In [ ]:
R = np.array(rows)
n_, kf, ke, ek, e, ep, eta = R.T

fig, axs = plt.subplots(1, 2, figsize=(12, 4.3))
ax = axs[0]
ax.semilogy(n_, ke, "o-", label=r"$\kappa_2(A)$ (high precision)")
ax.semilogy(n_, kf, "s--", alpha=.7, label=r"$\kappa_2(A)$ (float64 SVD)")
ax.axhline(1 / EPS, color="gray", ls=":", label=r"$1/\varepsilon$")
ax.set_xlabel("n"); ax.set_title("Frank matrix: condition number"); ax.legend()

ax = axs[1]
ax.semilogy(n_, ek, "k--", label=r"$\varepsilon_{mach}\,\kappa_2(A)$")
ax.semilogy(n_, np.maximum(e, 1e-17), "o-", label=r"$e_{rel}$ ($b$ exact)")
ax.semilogy(n_, ep, "^-", ms=4, label=r"$e_{rel}$ ($b$ perturbed at $\varepsilon$)")
ax.semilogy(n_, np.maximum(eta, 1e-18), "d:", ms=4, label=r"backward error $\eta_{A,b}$")
ax.axhline(1, color="gray", ls=":")
ax.set_xlabel("n"); ax.set_title("Relative error of the computed solution"); ax.legend(fontsize=8)
plt.show()

## Explanation through the backward error formula

Let $\hat x$ be the computed solution. GEPP is backward stable, so $\hat x$ is the exact solution of $(A+\delta A)\hat x=b+\delta b$ with
$$
\frac{\|\delta A\|}{\|A\|}=O(\varepsilon_{mach}),\qquad\frac{\|\delta b\|}{\|b\|}=O(\varepsilon_{mach}).
$$
Plugging this into
$$
\frac{\|\delta x\|}{\|\hat x\|}\le\kappa(A)\left(\frac{\|\delta A\|}{\|A\|}+\frac{\|\delta b\|}{\|A\|\|\hat x\|}\right)
\approx\kappa(A)\cdot O(\varepsilon_{mach})
$$
and observing what the experiment shows (three regimes):

1. **$\varepsilon\kappa_2\ll1$ (here $n\lesssim12$).** The relative error follows $\varepsilon\,\kappa_2(A)$: the computation loses $\approx\log_{10}\kappa_2(A)$ digits. The *backward* error column ($\eta$) stays at $10^{-17}$–$10^{-16}$ at all times: GEPP is doing its job; the loss of accuracy is entirely due to the conditioning of $A$. The ratio $e_{rel}/(\varepsilon\kappa_2)$ is usually between $10^{-2}$ and $1$: the bound is a worst case, but it captures the growth, and the observed error never exceeds it by more than a modest factor.
2. **$\varepsilon\kappa_2\approx1$ (here $n\approx17$–$18$).** The error is $O(10^{-1})$–$O(1)$: no significant digit survives.
3. **$\varepsilon\kappa_2\gg1$ ($n\gtrsim19$).** The bound is vacuous ($>1$) and the computed solution is meaningless: the relative error is $O(1)$ or larger, but no longer follows $\varepsilon\kappa_2$ (the error is now dominated by the specific structure of the elimination, not by the bound), and the column "$k_2$ float64" even stops growing because the SVD itself cannot resolve $\sigma_{\min}$.

Further observations:

* For $n=2$ the error is exactly $0$ and for $n=3,4,5$ it is essentially $0$ or $O(\varepsilon)$: the integer data are processed without any appreciable rounding.
* The two versions ($b$ exact / $b$ polluted at the $\varepsilon$ level) behave the same: the error is dominated by $\kappa$, not by *which* tiny perturbation hits the problem. This is exactly the content of the perturbation bound: **any** perturbation of size $\varepsilon$ is amplified by up to $\kappa$.
* The backward error remains tiny even when the forward error is $O(1)$. A small residual does **not** imply an accurate solution if $\kappa$ is large.

# Part 3 — Problem 3: "It is inadvisable to compute $A^{-1}$"

We compare two ways to solve $Ax=b$:

1. $x=A^{-1}b$, with $A^{-1}$ computed *properly* (solving $AX=I$ by GEPP, i.e. $n$ linear systems);
2. GEPP directly: $PA=LU$, then forward/back substitution.

**Cost.** One LU factorization costs $\tfrac23n^3$ flops. Forming $A^{-1}$ requires the LU plus $n$ pairs of triangular solves ($\approx2n^3$ in total counting the structure of the identity), after which $A^{-1}b$ costs another $2n^2$. So method 1 costs $\approx2n^3$ and method 2 costs $\approx\tfrac23n^3$: **3 times more expensive** (the exact count depends on how one exploits the zeros in $I$).

**Stability.** Even if $A^{-1}$ were known *exactly*, the product $\hat x=\mathrm{fl}(A^{-1}b)$ is a matrix–vector product with rounding errors, $\hat x=(A^{-1}+E)b$ with $|E|\le nu|A^{-1}|$. Therefore the residual is
$$
r=A\hat x-b=A E\,b,\qquad \|r\|\lesssim n\,u\,\|A\|\,\|A^{-1}\|\,\|b\|=n\,u\,\kappa(A)\,\|b\|,
$$
and
$$
\eta_{A,b}(\hat x)=\frac{\|r\|}{\|A\|\|\hat x\|+\|b\|}\ \text{ can be as large as }\ \approx\kappa(A)\,u,
$$
whereas for GEPP $\eta_{A,b}=O(u)$ (backward stable) regardless of $\kappa(A)$. The two methods have forward errors of the same order $\kappa u$, but the inverse method returns an $\hat x$ that is *not even the solution of a nearby problem*.

The matrix inverse also loses accuracy in forming $A^{-1}$ itself: the computed $\hat C\approx A^{-1}$ only satisfies $\|A\hat C-I\|\lesssim c\,u\,\kappa(A)$.

## Experiment design
Following the handout:

* Random $A\in\mathbb R^{n\times n}$ with $\kappa_2(A)=\alpha=u^{-1/2}\approx9\cdot10^7$ (so $\kappa u\approx10^{-8}$: the inverse method should have backward error $\approx10^{-8}$ in the worst case, while GEPP stays at $10^{-16}$).
* Generation (remark of the handout): $B$ Gaussian, $B=QR$, $A=Q^TDQ$, $D=\mathrm{diag}(\alpha,1,\dots,1)$. We use the sign-corrected $Q$ (Haar distributed). By construction $A$ is symmetric positive definite with $\|A\|_2=\alpha$ and $\kappa_2(A)=\alpha$.
* $x_0\sim N(0,I)$, $b=Ax_0$.
* For each system compute $\hat x_{inv}=\mathrm{inv}(A)\,b$ and $\hat x_{LU}$ (GEPP via `lu_factor`/`lu_solve`), and $\eta_{A,b}=\|r\|_2/(\|A\|_2\|\hat x\|_2+\|b\|_2)$.

In [ ]:
ALPHA = U ** -0.5
print(f"alpha = u^(-1/2) = {ALPHA:.3e}")

def rand_spd_cond(n, alpha, rng):
    '''A = Q^T D Q, D = diag(alpha,1,...,1): symmetric positive definite, kappa_2(A)=alpha, ||A||_2=alpha.'''
    Q, R = np.linalg.qr(rng.standard_normal((n, n)))
    Q = Q * np.sign(np.diag(R))
    D = np.ones(n); D[0] = alpha
    return Q.T @ (D[:, None] * Q)

# check the generator
rng = np.random.default_rng(0)
A = rand_spd_cond(25, ALPHA, rng)
print("kappa_2(A) =", np.linalg.cond(A), "  ||A||_2 =", np.linalg.norm(A, 2))

def backward_error(A, x, b, normA):
    return np.linalg.norm(A @ x - b) / (normA * np.linalg.norm(x) + np.linalg.norm(b))

def experiment(n, N, alpha=ALPHA, seed=0):
    rng = np.random.default_rng(seed)
    eta_inv, eta_lu, fwd_inv, fwd_lu = [], [], [], []
    for _ in range(N):
        A = rand_spd_cond(n, alpha, rng)
        x0 = rng.standard_normal(n)
        b = A @ x0
        x_inv = np.linalg.inv(A) @ b                         # method 1: explicit inverse (computed via GEPP)
        x_lu  = sla.lu_solve(sla.lu_factor(A), b)            # method 2: GEPP
        eta_inv.append(backward_error(A, x_inv, b, alpha))   # ||A||_2 = alpha by construction
        eta_lu.append(backward_error(A, x_lu, b, alpha))
        fwd_inv.append(np.linalg.norm(x_inv - x0) / np.linalg.norm(x0))
        fwd_lu.append(np.linalg.norm(x_lu - x0) / np.linalg.norm(x0))
    return map(np.array, (eta_inv, eta_lu, fwd_inv, fwd_lu))

results = {}
for n, N in [(25, 50), (50, 10_000)]:
    ei, el, fi, fl = experiment(n, N)
    results[(n, N)] = (ei, el, fi, fl)
    print(f"\n n = {n}, {N} systems,  kappa_2 = {ALPHA:.2e}")
    print(f"   backward error  inverse method : min = {ei.min():.2e}   max = {ei.max():.2e}   median = {np.median(ei):.2e}")
    print(f"   backward error  GEPP           : min = {el.min():.2e}   max = {el.max():.2e}   median = {np.median(el):.2e}")
    print(f"   forward error   inverse method : median = {np.median(fi):.2e}   max = {fi.max():.2e}")
    print(f"   forward error   GEPP           : median = {np.median(fl):.2e}   max = {fl.max():.2e}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, key in zip(axs, results):
    ei, el, fi, fl = results[key]
    ax.boxplot([np.log10(ei), np.log10(el)], labels=["inverse  $A^{-1}b$", "GEPP"])
    ax.axhline(np.log10(U), color="gray", ls=":"); ax.text(1.35, np.log10(U) + .2, "u", color="gray")
    ax.axhline(np.log10(U * ALPHA), color="gray", ls=":"); ax.text(1.35, np.log10(U * ALPHA) + .2, r"$u\kappa$", color="gray")
    ax.set_ylabel(r"$\log_{10}\,\eta_{A,b}$"); ax.set_title(f"n = {key[0]}, {key[1]} systems")
plt.show()

### Interpretation

* **GEPP:** $\eta_{A,b}$ is between $10^{-18}$ and a few $10^{-16}$, i.e. $O(u)$: backward stable, independent of the large $\kappa$.
* **Inverse method:** $\eta_{A,b}$ ranges over several orders of magnitude *above* $u$ and in the worst cases reaches $\approx\kappa u\approx10^{-8}$–$10^{-9}$, as predicted. The explicit inverse produces solutions whose residual is up to **eight orders of magnitude** larger.
* The **forward errors** of both methods are of the same order ($\approx\kappa u$): the loss of accuracy of the *answer* is unavoidable (it is the conditioning of the problem), but the inverse method additionally destroys the backward stability.

### How does the backward error of the inverse method depend on $\kappa$?
The bound says $\eta\lesssim\kappa u$; let us verify the scaling by sweeping $\kappa$.

In [ ]:
kap = 10.0 ** np.arange(1, 15)
mean_inv, mean_lu = [], []
for k in kap:
    ei, el, _, _ = experiment(30, 100, alpha=k, seed=5)
    mean_inv.append(ei.mean()); mean_lu.append(el.mean())

plt.loglog(kap, mean_inv, "o-", label="inverse method")
plt.loglog(kap, mean_lu, "s-", label="GEPP")
plt.loglog(kap, U * kap, "k--", label=r"$u\,\kappa$")
plt.axhline(U, color="gray", ls=":")
plt.xlabel(r"$\kappa_2(A)$"); plt.ylabel(r"mean backward error $\eta_{A,b}$"); plt.legend()
plt.title("Backward error vs. condition number (n=30)"); plt.show()

m = kap >= 1e3
print("log-log slope of eta_inverse vs kappa (kappa>=1e3):", np.polyfit(np.log10(kap[m]), np.log10(np.array(mean_inv)[m]), 1)[0])

The backward error of the inverse method grows **linearly with $\kappa$** (slope $\approx1$ in the log–log plot), exactly as the bound $\eta\lesssim\kappa u$ predicts, while GEPP stays flat at $u$.

### What if the inverse were computed exactly?
The handout stresses that the instability persists "even if $A^{-1}$ is computed exactly". We can test this: compute $A^{-1}$ in 60-digit arithmetic, round it to double precision (the best double-precision approximation of $A^{-1}$), and then multiply by $b$ in double precision.

In [ ]:
if HAVE_MP:
    mp.mp.dps = 60
    rng = np.random.default_rng(7)
    n, N = 25, 10
    eta_exact_inv = []
    for _ in range(N):
        A = rand_spd_cond(n, ALPHA, rng)
        x0 = rng.standard_normal(n); b = A @ x0
        Ainv_mp = mp.inverse(mp.matrix(A.tolist()))                     # (essentially) exact inverse of the stored A
        Ainv = np.array(Ainv_mp.tolist(), dtype=float)                  # rounded to double
        eta_exact_inv.append(backward_error(A, Ainv @ b, b, ALPHA))
    print("backward error with the correctly rounded exact inverse (n=25, 10 systems):")
    print("   min = %.2e   max = %.2e" % (min(eta_exact_inv), max(eta_exact_inv)))
    print("compare: GEPP ~ 1e-17..1e-16 ; inverse computed by GEPP:", "%.1e .. %.1e" % (results[(25, 50)][0].min(), results[(25, 50)][0].max()))
else:
    print("mpmath not available: experiment skipped")

Even a (rounded) *exact* inverse gives a backward error many orders of magnitude above $u$: the culprit is the matrix–vector product with $A^{-1}$ (a matrix with huge norm, $\|A^{-1}\|\approx1$ but cancellation-prone relative to $\|b\|$), exactly as in the analysis above. **Moral: solve with a factorization; never form $A^{-1}$ unless the inverse itself is the object of interest.**

## Timing: cost as a function of $n$
For $n=10,20,\dots,100$ we solve $10^3$ systems and compare the wall-clock time of

* `inv(A) @ b`,
* `lu_factor` + `lu_solve` (GEPP),
* `np.linalg.solve` (LAPACK `gesv`, i.e. the same GEPP without the Python overhead of two calls).

(For speed, a pool of 100 distinct matrices is generated and each is used 10 times; the timing is not affected, since only the dimension matters.)

In [ ]:
def timeit_systems(solver, As, bs, reps):
    t0 = time.perf_counter()
    for _ in range(reps):
        for A, b in zip(As, bs):
            solver(A, b)
    return time.perf_counter() - t0

solvers = {
    "inverse":        lambda A, b: np.linalg.inv(A) @ b,
    "GEPP (scipy)":   lambda A, b: sla.lu_solve(sla.lu_factor(A), b),
    "np.linalg.solve": lambda A, b: np.linalg.solve(A, b),
}

ns_t = np.arange(10, 101, 10)
times = {k: [] for k in solvers}
rng = np.random.default_rng(11)
for n in ns_t:
    As = [rand_spd_cond(int(n), ALPHA, rng) for _ in range(100)]
    bs = [rng.standard_normal(int(n)) for _ in range(100)]
    for name, f in solvers.items():
        timeit_systems(f, As[:5], bs[:5], 1)                      # warm-up
        times[name].append(timeit_systems(f, As, bs, 10))         # 1000 systems

print(f"{'n':>4} " + " ".join(f"{k:>16}" for k in solvers) + f" {'inv/GEPP':>10} {'inv/solve':>10}")
for i, n in enumerate(ns_t):
    print(f"{n:>4} " + " ".join(f"{times[k][i]:>16.4f}" for k in solvers)
          + f" {times['inverse'][i]/times['GEPP (scipy)'][i]:>10.2f} {times['inverse'][i]/times['np.linalg.solve'][i]:>10.2f}")

for k in solvers:
    plt.plot(ns_t, times[k], "o-", label=k)
plt.xlabel("n"); plt.ylabel("time for 1000 systems (s)"); plt.legend(); plt.title("Cost of the solvers"); plt.show()

### Interpretation of the timings

* The inverse method is slower than direct GEPP, and the gap **grows with $n$**, consistent with the flop counts $2n^3$ vs. $\tfrac23n^3$ (ratio 3 asymptotically).
* For small $n$ the ratio is not 3 (it can even be below 1 for the Python-level `lu_factor`+`lu_solve` pair): the cost is dominated by call overhead and memory traffic rather than flops. `np.linalg.solve` (a single LAPACK call) shows the true picture earlier. Flop counts predict the *asymptotics*, not the timings at tiny sizes.
* Also, performance ratios depend on BLAS/LAPACK implementation, number of threads and hardware, so the exact numbers will differ on each machine; the trend is what matters.

**Summary of Problem 3:** the inverse method is (i) $\approx3\times$ more expensive and (ii) numerically less stable: its backward error scales like $\kappa u$ instead of $u$.

# Part 4 — Problem 4: Estimating the condition number $\kappa_1(A)$

Computing $\kappa(A)=\|A\|\|A^{-1}\|$ explicitly requires $A^{-1}$, i.e. $O(n^3)$ work *on top of* the factorization — and we just saw that forming $A^{-1}$ is something to avoid. But $\|A\|_1$ is cheap ($O(n^2)$), so the whole difficulty is $\|A^{-1}\|_1$. We want an **estimate** $\omega\approx\|A^{-1}\|_1$ using **only the LU factors of $A$** (which we have anyway when solving $Ax=b$), at $O(n^2)$ cost: each iteration costs two triangular solves.

## 4(a) Hager's algorithm for $\|B\|_1$

### Idea
$\|B\|_1=\max_{\|x\|_1\le1}\|Bx\|_1$ is the maximum of the **convex** function $f(x)=\|Bx\|_1$ over the convex set $\{\|x\|_1\le1\}$ (a cross-polytope). A convex function attains its maximum at a **vertex**, i.e. at some $\pm e_j$; so $\|B\|_1=\max_j\|Be_j\|_1$ (max column sum, as we already knew). Trying all $e_j$ costs $O(n^2)$ *given $B$*, but we don't have $B=A^{-1}$ available. Hager's algorithm is a **gradient (steepest-ascent) method on the vertices**, which only needs products $Bx$ and $B^T\zeta$.

### The algorithm (Demmel, Alg. 2.5)
```
choose x with ||x||_1 = 1          (e.g. x_i = 1/n)
repeat
    w = B x ,  zeta = sign(w) ,  z = B^T zeta          # z^T = gradient of f at x
    if ||z||_inf <= z^T x:   return ||w||_1            # stop: x is a local maximum
    else  x = e_j  where |z_j| = ||z||_inf             # move to the best vertex
```

### Why does it work? (proof sketch)
Where $w$ has no zero entries, $f(x)=\zeta^TBx$ is locally linear with gradient $z=B^T\zeta$ (so $f$ is differentiable there). By convexity, for any $y$,
$$
f(y)\ge f(x)+z^T(y-x).
$$
* **Stopping test.** $\max_{\|y\|_1\le1}z^Ty=\|z\|_\infty$. If $\|z\|_\infty\le z^Tx$, then $z^T(y-x)\le0$ for every feasible $y$: no first-order ascent direction exists, $x$ is a local maximum of $f$ on the ball, and we return $f(x)=\|w\|_1\le\|B\|_1$.
* **Otherwise** pick $j$ with $|z_j|=\|z\|_\infty>z^Tx$. Since $f(e_j)=f(-e_j)$, take $y=\mathrm{sign}(z_j)e_j$: then $f(e_j)\ge f(x)+\bigl(\|z\|_\infty-z^Tx\bigr)>f(x)$. So $f$ **strictly increases** at every iteration, the iterates are vertices that never repeat, and the method terminates (in theory after at most $n$ steps; in practice after 2–3).

The returned value is always a **lower bound**: $\omega=\|Bx\|_1/\|x\|_1\le\|B\|_1$.

In [ ]:
def hager(Bmul, BTmul, n, maxit=5, higham=False, verbose=False):
    '''
    Hager's estimator of ||B||_1 (Demmel, Algorithm 2.5), accessing B only through
    the products Bmul(x) = B x  and  BTmul(y) = B^T y.

    Returns (omega, iterations) with  omega <= ||B||_1.
    higham=True adds the extra test with an alternating-sign vector (Higham's improvement used in LAPACK).
    '''
    x = np.full(n, 1.0 / n)                                  # ||x||_1 = 1
    omega = 0.0
    for it in range(1, maxit + 1):
        w = Bmul(x)
        zeta = np.where(w >= 0, 1.0, -1.0)                   # sign(w), with sign(0) = +1
        z = BTmul(zeta)
        omega = np.linalg.norm(w, 1)
        if verbose:
            print(f"it {it}: x={np.round(x,3)}  ||w||_1={omega:.5f}  ||z||_inf={np.linalg.norm(z, np.inf):.5f}  z^T x={z @ x:.5f}")
        if np.linalg.norm(z, np.inf) <= z @ x:               # local maximum reached
            break
        j = np.argmax(np.abs(z))                             # best vertex
        x = np.zeros(n); x[j] = 1.0
    if higham and n > 1:
        # alternating-sign vector with ||x||_1 = 3n/2: 2||Bx||_1/(3n) is again a lower bound for ||B||_1
        xa = (-1.0) ** np.arange(n) * (1 + np.arange(n) / (n - 1))
        omega = max(omega, 2 * np.linalg.norm(Bmul(xa), 1) / (3 * n))
    return omega, it

### A worked example (verbose)
For a $5\times5$ matrix we print every iteration: $x$, $\|w\|_1$ (the current lower bound), the gradient test $\|z\|_\infty\le z^Tx$.

In [ ]:
rng = np.random.default_rng(4)
B = rng.standard_normal((5, 5))
omega, its = hager(lambda x: B @ x, lambda y: B.T @ y, 5, verbose=True)
print(f"\nHager estimate = {omega:.6f}   exact ||B||_1 = {np.linalg.norm(B, 1):.6f}   (iterations: {its})")
print("column sums:", np.abs(B).sum(axis=0).round(4))

### Test: quality of the estimate for random matrices
We test with 500 Gaussian matrices of random size $5\le n\le59$ and record: (i) the ratio $\omega/\|B\|_1$ (always $\le1$), (ii) how often the estimate is *exact*, (iii) the number of iterations (the handout says "typically it stops after 2 iterations").

In [ ]:
rng = np.random.default_rng(5)
ratio, iters = [], []
for _ in range(500):
    n = rng.integers(5, 60)
    B = rng.standard_normal((n, n))
    w, it = hager(lambda x: B @ x, lambda y: B.T @ y, n)
    ratio.append(w / np.linalg.norm(B, 1)); iters.append(it)
ratio, iters = np.array(ratio), np.array(iters)

print("always a lower bound (ratio <= 1 + 1e-12):", bool(np.all(ratio <= 1 + 1e-12)))
print(f"ratio  min = {ratio.min():.3f}   median = {np.median(ratio):.3f}   fraction exact (>0.999) = {np.mean(ratio > 0.999):.2%}")
print("iterations (histogram 1,2,3,4,5):", np.bincount(iters, minlength=6)[1:])

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
axs[0].hist(ratio, bins=25); axs[0].set_xlabel(r"$\omega/\|B\|_1$"); axs[0].set_title("Quality of Hager's estimate (Gaussian B)")
axs[1].bar(np.arange(1, 6), np.bincount(iters, minlength=6)[1:]); axs[1].set_xlabel("iterations"); axs[1].set_title("Iterations needed")
plt.show()

**Interpretation.** The estimate is *always* a lower bound, it needs 2 iterations in the vast majority of cases (the first one is only used to verify the stopping test at the starting point), and for a Gaussian matrix with *flat* column sums it is accurate within a factor of $\approx2$ in the worst cases seen here. For $\kappa$ estimation, a factor of 2–3 is perfectly acceptable: only the *order of magnitude* of $\kappa$ matters.

## 4(b) Estimating $\|A^{-1}\|_1$ and the condition number

Take $B=A^{-1}$. We never form $B$; instead
$$
w=Bx\iff Aw=x,\qquad z=B^T\zeta\iff A^Tz=\zeta .
$$
With one LU factorization $PA=LU$ ($\tfrac23n^3$ flops, the same one we need to solve $Ax=b$) each product costs two triangular solves, $O(n^2)$. Note that $A^Tz=\zeta$ is solved with the **same** factors: $A^T=U^TL^TP$, i.e. `lu_solve(lu, zeta, trans=1)`.

Then
$$
\hat\kappa_1(A)=\|A\|_1\cdot\omega,\qquad \omega\approx\|A^{-1}\|_1,\qquad \hat\kappa_1(A)\le\kappa_1(A).
$$

In [ ]:
def condest1(A, lu=None, maxit=5, higham=False):
    '''Estimate of kappa_1(A) = ||A||_1 ||A^{-1}||_1 using only LU solves (no inverse).'''
    n = A.shape[0]
    if lu is None:
        lu = sla.lu_factor(A)                                            # GEPP: PA = LU
    Binv  = lambda x: sla.lu_solve(lu, x)                                # w = A^{-1} x   <=>  A w = x
    BinvT = lambda y: sla.lu_solve(lu, y, trans=1)                       # z = A^{-T} y   <=>  A^T z = y
    omega, its = hager(Binv, BinvT, n, maxit=maxit, higham=higham)
    return np.linalg.norm(A, 1) * omega, its

# small test
rng = np.random.default_rng(6)
A = rng.standard_normal((6, 6))
est, its = condest1(A)
print("estimate :", est, f"({its} iterations)")
print("direct   :", np.linalg.norm(A, 1) * np.linalg.norm(np.linalg.inv(A), 1))

### Validation experiment (as requested)
For 1000 random matrices with dimensions $10\le n\le90$ and $10^3\le\kappa_2(A)\le10^{16}$ we record the ratio
$$
\rho=\frac{\hat\kappa_1(A)\ \text{(Hager)}}{\kappa_1(A)\ \text{(direct: }\|A\|_1\|A^{-1}\|_1\text{ with the explicit inverse)}} .
$$

* Matrices are generated with a **general** construction $A=U\,\mathrm{diag}(\sigma)\,V^T$ ($U,V$ random orthogonal, $\sigma_{\max}=1$, $\sigma_{\min}=1/\kappa_2$, other singular values log-uniform in between), with $\log_{10}\kappa_2$ uniform in $[3,16]$. (The generator $Q^TDQ$ of Problem 3 yields *symmetric* matrices with a single large singular value, which would test the transposed solve only trivially.)
* We also record two reference estimates: Hager with Higham's extra test, and the **LAPACK** routine `dgecon` (which implements Higham's improvement of Hager's algorithm), via `scipy.linalg.lapack.dgecon`.
* **Caveat:** for $\kappa_2\gtrsim10^{14}$ the explicit inverse is itself inaccurate (we know it from Problem 3), so the "direct" value is not an exact reference there; ratios slightly above 1 are possible and reflect errors of the direct computation, not of Hager's estimator.

In [ ]:
from scipy.linalg import lapack

rng = np.random.default_rng(2024)
n_mat = 1000
recs = []                                      # (n, kappa2, rho_hager, rho_higham, rho_lapack, iterations)
for _ in range(n_mat):
    n = int(rng.integers(10, 91))
    kappa2 = 10 ** rng.uniform(3, 16)
    A = rand_matrix_cond(n, kappa2, rng)

    lu_piv = sla.lu_factor(A)
    normA = np.linalg.norm(A, 1)
    direct = normA * np.linalg.norm(np.linalg.inv(A), 1)

    est, its = condest1(A, lu=lu_piv)
    est_h, _ = condest1(A, lu=lu_piv, higham=True)
    rcond, info = lapack.dgecon(lu_piv[0], normA, norm="1")
    recs.append((n, kappa2, est / direct, est_h / direct, (1 / rcond) / direct, its))

recs = np.array(recs)
rho, rho_h, rho_l, its = recs[:, 2], recs[:, 3], recs[:, 4], recs[:, 5].astype(int)

def summary(name, r):
    print(f"{name:<24} min = {r.min():.3f}  5% = {np.quantile(r, .05):.3f}  median = {np.median(r):.3f}  max = {r.max():.4f}   "
          f"P(rho>0.5) = {np.mean(r > .5):.1%}   P(rho>0.9) = {np.mean(r > .9):.1%}")
summary("Hager (basic)", rho)
summary("Hager + Higham test", rho_h)
summary("LAPACK dgecon", rho_l)
print("iterations of Hager (1..5):", np.bincount(its, minlength=6)[1:])

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].hist(rho, bins=40, alpha=.8, label="Hager")
axs[0].hist(rho_h, bins=40, alpha=.5, label="Hager + Higham")
axs[0].set_xlabel(r"$\hat\kappa_1/\kappa_1$"); axs[0].set_title("Ratio estimate / direct"); axs[0].legend()

sc = axs[1].scatter(recs[:, 1], rho, s=6, c=recs[:, 0], cmap="viridis")
axs[1].set_xscale("log"); axs[1].set_xlabel(r"$\kappa_2(A)$"); axs[1].set_ylabel(r"$\hat\kappa_1/\kappa_1$"); axs[1].set_title("Ratio vs condition number")
axs[1].axhline(1, color="k", lw=.8); plt.colorbar(sc, ax=axs[1], label="n")

axs[2].scatter(recs[:, 0], rho, s=6)
axs[2].set_xlabel("n"); axs[2].set_ylabel(r"$\hat\kappa_1/\kappa_1$"); axs[2].set_title("Ratio vs dimension"); axs[2].axhline(1, color="k", lw=.8)
plt.show()

### Interpretation

* Hager's estimator is a **lower bound** up to the accuracy of the reference: the maximum observed ratio is $1.00\ldots$, with excess (if any) only for huge $\kappa_2$, where the explicit inverse is itself polluted by rounding.
* The *median ratio is close to 1* and almost all matrices have $\hat\kappa_1\ge\kappa_1/2$ — so the estimate gives the correct order of magnitude, which is all we need to decide how many digits can be trusted ($\approx16-\log_{10}\kappa$).
* The quality does not degrade with $n$ or with $\kappa_2$ (the scatter plots have no trend): the estimator is **robust**.
* The extra test of Higham (alternating-sign vector) and the LAPACK routine improve the rare bad cases; this is why LAPACK/Matlab (`dgecon`, `condest`) use them.
* The algorithm needs, on top of the LU factorization, only $\approx4$ triangular solves ($2$ iterations $\times$ $2$ solves), i.e. $O(n^2)$ work, compared with $O(n^3)$ for the explicit inverse. We measure it below.

### Cost: estimating vs. inverting
We time (i) $\|A\|_1\|A^{-1}\|_1$ with the explicit inverse, (ii) the Hager estimator *given* the LU factorization, (iii) LU + Hager.

In [ ]:
rng = np.random.default_rng(8)
print(f"{'n':>5} {'inv+norm (s)':>14} {'LU (s)':>10} {'Hager given LU (s)':>20} {'speed-up (inv / Hager)':>24}")
for n in [100, 200, 400, 800, 1600]:
    A = rng.standard_normal((n, n)) + n * np.eye(n)
    t0 = time.perf_counter(); direct = np.linalg.norm(A, 1) * np.linalg.norm(np.linalg.inv(A), 1); t_inv = time.perf_counter() - t0
    t0 = time.perf_counter(); lu = sla.lu_factor(A); t_lu = time.perf_counter() - t0
    t0 = time.perf_counter(); est, _ = condest1(A, lu=lu); t_h = time.perf_counter() - t0
    print(f"{n:>5} {t_inv:>14.4f} {t_lu:>10.4f} {t_h:>20.5f} {t_inv/t_h:>24.1f}      (ratio est/direct = {est/direct:.3f})")

Given the LU factorization the estimate is essentially free ($O(n^2)$ vs $O(n^3)$), and the gap widens with $n$ — in agreement with the complexity analysis. This is what makes condition estimation a standard by-product of solvers: LAPACK's `dgesvx` returns `rcond` at negligible cost.

# Part 5 — Summary

| Topic | Take-away |
|---|---|
| **Theory** | $\dfrac{\|\delta x\|}{\|\hat x\|}\le\kappa(A)\Bigl(\dfrac{\|\delta A\|}{\|A\|}+\dfrac{\|\delta b\|}{\|A\|\|\hat x\|}\Bigr)$, sharp; $1/\kappa_2$ = relative distance to singularity; digits lost $\approx\log_{10}\kappa$. |
| **Problem 1(a)** | Vandermonde on equispaced nodes: $\kappa_2$ grows exponentially (roughly a factor $\approx3$ per node); $>10^{16}$ already at $n\approx20$. Basis/nodes matter: Chebyshev basis at Chebyshev nodes has $\kappa_2=O(1)$. |
| **Problem 1(b)** | Hilbert: $\kappa_2(H_n)\sim C(1+\sqrt2)^{4n}/\sqrt n\approx e^{3.5255n}$; slope of $\ln\kappa_2$ fitted $\approx3.51$ — Szegő's $e^{3.5n}$ confirmed. float64 SVD saturates at $\approx10^{17}$–$10^{19}$ (numerically singular for $n\gtrsim13$). |
| **Problem 2** | Frank matrix: relative error $\approx\varepsilon\kappa_2$ (pessimistic by $10^{-2}$–$1$) while $\varepsilon\kappa\ll1$; $O(1)$ errors once $\kappa\gtrsim1/\varepsilon$; backward error remains $O(u)$ throughout. $\det A=1$ is irrelevant. |
| **Problem 3** | Explicit inverse: $\approx3\times$ more flops and backward error $\sim\kappa u$ (vs $u$ for GEPP) — even with an exactly computed inverse. Forward errors are comparable. |
| **Problem 4** | Hager's algorithm estimates $\|A^{-1}\|_1$ from LU solves only ($O(n^2)$): always a lower bound, usually within a small factor (median ratio close to 1), 2 iterations typically. Higham's improvement is what LAPACK uses. |

### Further reading
* J. Demmel, *Applied Numerical Linear Algebra*, SIAM, 1997 — §2.2–2.4 (norms, perturbation theory, condition estimation, Alg. 2.5).
* N. J. Higham, *Accuracy and Stability of Numerical Algorithms*, 2nd ed., SIAM, 2002 — Ch. 7 (perturbation theory), Ch. 9 (LU), Ch. 14 (condition number estimation), Ch. 28 (Vandermonde), Ch. 14 on Hilbert.
* L. N. Trefethen, D. Bau, *Numerical Linear Algebra*, SIAM, 1997 — Lectures 12–14 (conditioning, stability).
* W. W. Hager, "Condition estimates", *SIAM J. Sci. Stat. Comput.* 5 (1984).
* N. J. Higham, "FORTRAN codes for estimating the one-norm of a real or complex matrix, with applications to condition estimation", *ACM TOMS* 14 (1988).
* G. Szegő, "On some Hermitian forms associated with two given curves of the complex plane", *Trans. AMS* 40 (1936) (Hilbert matrix asymptotics; see also J. Todd, 1954).

### Suggested extensions
1. Repeat the Vandermonde experiment with $\kappa_\infty$ and $\kappa_1$ and check the norm equivalences $\tfrac1n\kappa_2\le\kappa_1\le n\kappa_2$.
2. Use *iterative refinement* to improve the Frank-matrix solutions when $\kappa\varepsilon<1$, and see that it fixes the forward error only if the residual is computed in higher precision.
3. Implement the full Higham–Tisseur block algorithm for $\|A\|_1$ estimation (`scipy.sparse.linalg.onenormest`) and compare it with Hager's.